In [2]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import os
from transformers import get_cosine_schedule_with_warmup
import torch.autograd as autograd

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7")
model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7",
                                                           num_labels=3,
                                                          trust_remote_code=True)
# model = nn.DataParallel(model)
model = model.to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mdeberta-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
df = pd.read_csv(r'/kaggle/input/datasets/popchic/kaaagle/train.csv')
# for col in ["premise", "hypothesis"]:
#     df[col] = df[col].astype(str).str.strip()
#     df.loc[df[col] == "nan", col] = ""  # pandas иногда пишет "nan" как строку

# # Проверка
# assert not (df["premise"] == "").any(), "❌ Есть пустые premise"
# assert not (df["hypothesis"] == "").any(), "❌ Есть пустые hypothesis"
# print("✅ Данные строго валидны")

train_df, temp_df = train_test_split(
    df, test_size=0.25, stratify=df["language"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.4, stratify=temp_df["language"], random_state=42
)

In [5]:
lang_counts = Counter(train_df["language"])
max_count = max(lang_counts.values())

balanced_rows = []
for lang, count in lang_counts.items():
    lang_subset = train_df[train_df["language"] == lang]

    if count < max_count:
        repeats = int(np.ceil(max_count / count))
        balanced_rows.append(pd.concat([lang_subset] * repeats, ignore_index=True))
    else:
        balanced_rows.append(lang_subset.sample(n=max_count, random_state=42))

train_balanced_df = pd.concat(balanced_rows, ignore_index=True)

In [6]:
class NLIDataset(Dataset):
    def __init__(self, premise, hypothesis, label = None):
        self.premise = list(premise)
        self.hypothesis = list(hypothesis)
        self.label = list(label) if label is not None else label 
        
    def __len__(self): 
        return len(self.premise)
    def __getitem__(self, i):
        enc = tokenizer(self.premise[i], self.hypothesis[i], truncation=True,
                        padding="max_length", max_length=128, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0), 
                "label": torch.tensor(self.label[i], dtype=torch.long) if self.label else None}

BATCH_SIZE = 16

train_loader = DataLoader(NLIDataset(train_balanced_df["premise"], train_balanced_df["hypothesis"], train_balanced_df["label"]),
                         batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(NLIDataset(val_df["premise"], val_df["hypothesis"], val_df["label"]),
                       batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(NLIDataset(test_df["premise"], test_df["hypothesis"], test_df["label"]),
                         batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

In [ ]:
autograd.set_detect_anomaly(True)

epochs = 3
model = model.to(torch.float32)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)

best_val_acc = 0.0
patience = 2
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    train_preds, train_labels_list = [], []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} | Train"):
        optimizer.zero_grad()
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids=ids, attention_mask=mask).logits
        train_preds.append(torch.argmax(logits, dim=-1).cpu())
        train_labels_list.append(labels.cpu())
        # print(batch)
        # if epoch == 0 and (batch_num == 0 or batch_num==1):
        #     print("\n🔍 Диагностика первого батча:")
        #     print(f"  labels: {labels.unique()}")
        #     print(f"  input_ids NaN: {torch.isnan(batch['input_ids']).any()}")
        #     print(f"  attention_mask все нули: {(batch['attention_mask'].sum(dim=1)==0).any()}")
        # print(logits)
        if torch.isnan(logits).any():
            print("NaN в logits! Пропускаем батч.")
            nan_detected = True
            continue
        loss = criterion(logits, labels)
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    train_acc = accuracy_score(torch.cat(train_labels_list), torch.cat(train_preds))
    
    model.eval()
    total_val_loss = 0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch_val {epoch+1}"):
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            
            loss = criterion(logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=-1)
            val_preds.append(preds.cpu())
            val_labels.append(labels.cpu())
    
    val_acc = accuracy_score(torch.cat(val_labels), torch.cat(val_preds))
    print(f"Epoch {epoch+1} | "
          f"Train Loss: {total_train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {total_val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        print("Новая лучшая модель сохранена в память!")

Epoch 1 | Train:  75%|███████▌  | 3729/4970 [58:12<19:28,  1.06it/s]  

In [ ]:
if best_model_state:
    model.load_state_dict(best_model_state)
    print("Загружены лучшие веса модели для тестирования.")
else:
    print("Модель не улучшилась. Используются веса последней эпохи.")

model.eval()
arr_ans = []
arr_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids=ids, attention_mask=mask).logits
        
        preds = torch.argmax(logits, dim=-1).cpu()
        
        arr_labels.append(labels)
        arr_ans.append(preds)

y_true = torch.cat(arr_labels).cpu().numpy()
y_pred = torch.cat(arr_ans).cpu().numpy()
test_acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {test_acc:.4f}")